# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 5: *Feature Interaction Analysis*
##### Version Number: 2.0
---
### Contents  
> 1. *Load Data*
> 2. *Split and Scale Data*
> 3. *Feature Interaction Analysis*
> 4. *Export Data*
---
### Notes
---
### Inputs
- `final.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `X`,`y` - for future modeling
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

# Function to print a custom format correlation heatmap
from src.plot_utils import correlation_map

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions

# A space saving function to rank interactions
from src.data_utils import rank_interactions_by_correlation

# Function to calculate dryness index and return a dataframe
from src.data_utils import calculate_dryness_index

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize

# Modeling prep
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import PolynomialFeatures

# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

## 1. Load Data

In [3]:
# Load cleaned main dataset
final = pd.read_csv("../data/processed/trimmed.csv")

In [4]:
final

,Unnamed: 0,Sample_ID,Date,Sample_Elevation,Region_ID,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Avg Air Temp (F),...,Total Population,density,Mean Income,Target,Days Without Rain,2-Year Avg Fires,Avg Air Temp (F) 7 Day Avg,Precip (in) 7 Day Avg,Avg Rel Hum (%) 7 Day Avg,Avg Wind Speed (mph) 7 Day Avg
0,0,72,2021-12-23,68.172875,4,0.04,0.00,446.0,10.7,59.2,...,152682,109.889018,75449,0,1,37.0,43.828571,0.052857,87.142857,3.885714
1,1,159,2021-10-06,1484.432861,1,0.06,0.00,237.0,9.0,50.8,...,42905,6.834303,65853,0,1,245.0,55.371429,0.005714,51.714286,3.300000
2,2,19,2018-10-16,282.537201,6,0.15,0.00,449.0,9.5,59.2,...,2492442,345.861225,93563,0,13,153.0,62.814286,0.000000,56.571429,4.214286
3,3,54,2021-08-23,1138.741699,6,0.33,0.00,633.0,9.2,90.6,...,2195611,109.468892,85327,0,24,325.5,89.914286,0.000000,27.000000,6.657143
4,4,96,2023-03-22,3330.114014,4,0.07,0.11,273.0,9.6,50.6,...,479468,99.387673,72092,0,0,0.0,55.014286,0.154286,76.714286,3.328571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22141,22141,173,2021-08-01,1618.552856,1,0.25,0.00,622.0,10.4,71.9,...,8500,2.169602,59138,2,5,325.5,71.314286,0.005714,37.714286,4.257143
22142,22142,173,2022-09-02,1618.552856,1,0.31,0.00,602.0,5.7,72.8,...,8500,2.169602,59138,2,15,285.0,69.785714,0.000000,24.857143,4.928571
22143,22143,173,2024-06-26,1618.552856,1,0.31,0.00,718.0,10.0,69.2,...,8500,2.169602,59138,1,23,45.5,67.385714,0.000000,44.000000,4.371429
22144,22144,173,2024-07-08,1618.552856,1,0.29,0.00,711.0,9.5,75.8,...,8500,2.169602,59138,1,35,189.5,71.257143,0.000000,34.285714,3.300000


---

## 2. Split and Scale Data

In [5]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target']

y = final['Target']
X = final.drop(columns=text_columns)
details = final[text_columns]

In [6]:
X

,Unnamed: 0,Sample_Elevation,Region_ID,ETo (in),Precip (in),Sol Rad (Ly/day),Avg Vap Pres (mBars),Avg Air Temp (F),Avg Rel Hum (%),Avg Wind Speed (mph),...,Season,Total Population,density,Mean Income,Days Without Rain,2-Year Avg Fires,Avg Air Temp (F) 7 Day Avg,Precip (in) 7 Day Avg,Avg Rel Hum (%) 7 Day Avg,Avg Wind Speed (mph) 7 Day Avg
0,0,68.172875,4,0.04,0.00,446.0,10.7,59.2,63.0,3.8,...,0,152682,109.889018,75449,1,37.0,43.828571,0.052857,87.142857,3.885714
1,1,1484.432861,1,0.06,0.00,237.0,9.0,50.8,71.0,3.8,...,3,42905,6.834303,65853,1,245.0,55.371429,0.005714,51.714286,3.300000
2,2,282.537201,6,0.15,0.00,449.0,9.5,59.2,55.0,3.9,...,3,2492442,345.861225,93563,13,153.0,62.814286,0.000000,56.571429,4.214286
3,3,1138.741699,6,0.33,0.00,633.0,9.2,90.6,19.0,5.9,...,2,2195611,109.468892,85327,24,325.5,89.914286,0.000000,27.000000,6.657143
4,4,3330.114014,4,0.07,0.11,273.0,9.6,50.6,76.0,4.1,...,1,479468,99.387673,72092,0,0.0,55.014286,0.154286,76.714286,3.328571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22141,22141,1618.552856,1,0.25,0.00,622.0,10.4,71.9,39.0,4.1,...,2,8500,2.169602,59138,5,325.5,71.314286,0.005714,37.714286,4.257143
22142,22142,1618.552856,1,0.31,0.00,602.0,5.7,72.8,21.0,7.6,...,3,8500,2.169602,59138,15,285.0,69.785714,0.000000,24.857143,4.928571
22143,22143,1618.552856,1,0.31,0.00,718.0,10.0,69.2,41.0,7.4,...,2,8500,2.169602,59138,23,45.5,67.385714,0.000000,44.000000,4.371429
22144,22144,1618.552856,1,0.29,0.00,711.0,9.5,75.8,31.0,3.4,...,2,8500,2.169602,59138,35,189.5,71.257143,0.000000,34.285714,3.300000


Scale all datasets and save back to dataframe format. MinMax Scaler used because majority of values are > 0

In [7]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

X_scaled = X

---

## 3. Interaction Correlation analysis

For all independent variables create a new dataframe containing every possible interaction of 2nd degree

In [8]:
inter_X = create_2nd_degree_interactions(X_scaled)

Combine interaction terms with single variables for correlation analysis

In [9]:
inter_X_combined = pd.concat([X_scaled, inter_X], axis=1)

In [10]:
correlation_results = rank_interactions_by_correlation(inter_X_combined, y)

C:\Users\dusti\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\dusti\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [11]:
correlation_results.head(20)

,Feature,Correlation
0,Unnamed: 0 x Avg Air Temp (F) 7 Day Avg,0.836225
1,Unnamed: 0 x Avg Soil Temp (F),0.829184
2,Unnamed: 0 x Avg Air Temp (F),0.828501
3,Unnamed: 0 x Season,0.827426
4,Unnamed: 0 x ETo (in),0.769085
5,Unnamed: 0 x Sol Rad (Ly/day),0.768260
6,Unnamed: 0,0.766976
7,Unnamed: 0 x Avg Vap Pres (mBars),0.731877
8,Unnamed: 0 x 2-Year Avg Fires,0.639567
9,Unnamed: 0 x Avg Wind Speed (mph) 7 Day Avg,0.574110


In [12]:
keep = correlation_results.head(40)['Feature'].tolist()

Keep top 20 results

In [13]:
top_20 = inter_X_combined[keep]

# Recompute correlation matrix
corr_matrix = top_20.corr()

to_drop = set()
cols = corr_matrix.columns

for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if corr_matrix.iloc[i, j] > 0.9:
            col_to_drop = cols[j]  # drop the later column in the pair
            to_drop.add(col_to_drop)

df_reduced = top_20.drop(columns=list(to_drop))

df_reduced

,Unnamed: 0 x Avg Air Temp (F) 7 Day Avg,Unnamed: 0 x Season,Unnamed: 0 x Avg Vap Pres (mBars),Unnamed: 0 x 2-Year Avg Fires,Unnamed: 0 x Avg Wind Speed (mph) 7 Day Avg,ETo (in) x Season,Unnamed: 0 x Avg Rel Hum (%) 7 Day Avg,Unnamed: 0 x Avg Wind Speed (mph),Avg Air Temp (F) 7 Day Avg,ETo (in) x Avg Soil Temp (F),Avg Vap Pres (mBars) x Season,ETo (in) x Avg Vap Pres (mBars),Unnamed: 0 x Mean Income,ETo (in) x 2-Year Avg Fires
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.327485,0.032934,0.000000,0.017753,0.000000,0.005253
1,0.000020,0.000045,0.000008,0.000019,0.000007,0.122449,0.000023,0.000005,0.451831,0.044056,0.182927,0.022399,0.000002,0.052174
2,0.000048,0.000090,0.000017,0.000024,0.000020,0.306122,0.000051,0.000010,0.532010,0.122692,0.193089,0.059109,0.000025,0.081455
3,0.000112,0.000090,0.000025,0.000077,0.000053,0.448980,0.000036,0.000025,0.823946,0.441853,0.124661,0.125933,0.000029,0.381242
4,0.000081,0.000060,0.000035,0.000000,0.000029,0.047619,0.000139,0.000021,0.447984,0.044785,0.065041,0.027875,0.000019,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22141,0.623464,0.666546,0.211344,0.565985,0.224369,0.340136,0.376184,0.116520,0.623576,0.258476,0.140921,0.107848,0.000000,0.288820
22142,0.607028,0.999865,0.115838,0.495585,0.270633,0.632653,0.247463,0.248087,0.607110,0.306284,0.115854,0.073295,0.000000,0.313576
22143,0.581203,0.666606,0.203234,0.079123,0.232262,0.421769,0.439159,0.240580,0.581256,0.289548,0.135501,0.128588,0.000000,0.050062
22144,0.622933,0.666637,0.193081,0.329550,0.158457,0.394558,0.341902,0.090221,0.622961,0.263039,0.128726,0.114277,0.000000,0.195049


In [14]:
X = df_reduced

---

## 4. Export Data

In [15]:
X.to_csv('../data/processed/X.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
